# SEC Hyperscaler + Secondaries Scanner — token-free (Colab)

Pulls **real SEC EDGAR data** (XBRL structured facts + full-text search) directly from the source — no key, no
token, no LLM calls. Built to operationalize the vault's fragility discipline: instead of waiting for a WSJ
piece to surface a footnote (the Google/SpaceX $94.1B stake — the paper had to do that for us on 2026-07-23),
this pulls the same filings on demand and flags the things the vault has been manually digging for all week.

**Three buckets, matching the quality-ladder razor:**
- **HYPERSCALERS** (cash-rich core): GOOGL, MSFT, AMZN, META
- **SECONDARIES** (chip/memory "sellers"): NVDA, AMD, INTC, MU, AVGO, TSM
- **PERIPHERY** (levered/neocloud, context only): ORCL, CRWV

**What it checks, per the vault's own threads:**
1. **The depreciation-schedule test** (Dowd/Goldman claim, 2026-07-24) — implied useful life = PP&E ÷ annualized
   depreciation. Lengthening while capex accelerates = the "artificial earnings" flag, made falsifiable.
2. **The off-balance-sheet commitments tracker** (the "$811B" / Beignet thread) — `LongTermPurchaseCommitmentAmount`
   and lease liabilities, trended over quarters.
3. **The unrealized-equity-gains catcher** (the SpaceX-stake mechanism) — `EquitySecuritiesFvNiGainLoss`, so the
   next one shows up here before a reporter has to find it.
4. **The FCF-proxy** (the "Google's first negative-cash-flow quarter" Facebook-chart thread) — operating cash
   flow minus capex.
5. **Secondaries fundamentals** — revenue, gross margin, inventory trend (memory/chip cycle read).
6. **Ad-hoc full-text search** — the Schedule-D-style "go find the receipt" tool: search any keyword (a
   counterparty name, "special purpose entity", "guarantee", "take-or-pay") across recent filings.

Run cells top to bottom. Every network call is wrapped defensively — a missing tag prints `N/A`, it never
crashes the notebook. Re-run any single cell any time; nothing is cached beyond the session.


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p])
try:
    import pandas as pd, requests
except Exception:
    _pip('pandas'); _pip('requests')
    import pandas as pd, requests
import time, json
from datetime import datetime

# SEC requires a descriptive User-Agent (name + contact) on every request, or it 403s/429s.
# Edit the email below if you want — SEC doesn't validate it, but keep the format (name + contact).
HEADERS = {'User-Agent': 'INMA-Research-Vault research-contact@example.com'}
SLEEP = 0.15  # stay well under SEC's ~10 req/sec limit across all the calls this notebook makes

def _get(url, params=None, timeout=15):
    """Defensive GET: never raises, returns None on any failure."""
    try:
        r = requests.get(url, headers=HEADERS, params=params, timeout=timeout)
        time.sleep(SLEEP)
        if r.status_code != 200:
            return None
        return r.json()
    except Exception:
        return None

print('Fetching SEC ticker->CIK map (one-time, ~10,400 companies)...')
_TICKER_MAP = _get('https://www.sec.gov/files/company_tickers.json')
if _TICKER_MAP is None:
    print('⚠️ Could not reach SEC — check internet connection / try re-running this cell.')
    TICKER_TO_CIK = {}
else:
    TICKER_TO_CIK = {v['ticker'].upper(): str(v['cik_str']).zfill(10) for v in _TICKER_MAP.values()}
    print(f'  Loaded {len(TICKER_TO_CIK)} tickers.')

def cik_for(ticker):
    return TICKER_TO_CIK.get(ticker.upper())


## Watchlist — the three buckets

In [ ]:
HYPERSCALERS = ['GOOGL', 'MSFT', 'AMZN', 'META']
SECONDARIES  = ['NVDA', 'AMD', 'INTC', 'MU', 'AVGO', 'TSM']
PERIPHERY    = ['ORCL', 'CRWV']  # levered/neocloud — context, not core

ALL_TICKERS = HYPERSCALERS + SECONDARIES + PERIPHERY

print('Resolving CIKs...')
for t in ALL_TICKERS:
    c = cik_for(t)
    print(f'  {t:6s} -> CIK {c}' if c else f'  {t:6s} -> NOT FOUND (ticker may differ on EDGAR, e.g. class shares)')


## XBRL concept puller

`companyconcept` returns the full historical time series for ONE tag. Tag names aren't perfectly standardized
across companies/years, so each concept below tries a short list of candidate tag names and uses the first
that returns data.

In [ ]:
def get_concept(cik, tag):
    """Pull one XBRL us-gaap concept's full history for a company. Returns a DataFrame or None."""
    if cik is None:
        return None
    d = _get(f'https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{tag}.json')
    if d is None or 'units' not in d or 'USD' not in d['units']:
        return None
    df = pd.DataFrame(d['units']['USD'])
    if df.empty:
        return None
    df['end'] = pd.to_datetime(df['end'])
    df = df.sort_values('end')
    return df

def get_concept_any(cik, candidate_tags):
    """Try each candidate tag in order; return (tag_used, DataFrame) for the first that has data."""
    for tag in candidate_tags:
        df = get_concept(cik, tag)
        if df is not None and len(df) > 0:
            return tag, df
    return None, None

def latest_annualized(df, period_days_min=80):
    """For a flow (income-statement/cash-flow) concept, get the most recent quarterly-ish value, annualized.
    Filters to entries that look like a single quarter (roughly 80-100 days) to avoid mixing YTD and Q figures."""
    if df is None or 'start' not in df.columns:
        return None
    d2 = df.dropna(subset=['start']).copy()
    d2['start'] = pd.to_datetime(d2['start'])
    d2['days'] = (d2['end'] - d2['start']).dt.days
    q = d2[(d2['days'] >= period_days_min) & (d2['days'] <= 100)]
    if q.empty:
        return None
    row = q.sort_values('end').iloc[-1]
    return row['val'] * 4, row['end']

# Candidate tag lists per concept (order = preference)
TAGS_PPE          = ['PropertyPlantAndEquipmentNet']
TAGS_DEPRECIATION = ['Depreciation', 'DepreciationDepletionAndAmortization', 'DepreciationAmortizationAndAccretionNet']
TAGS_CAPEX        = ['PaymentsToAcquirePropertyPlantAndEquipment', 'PaymentsForCapitalImprovements']
TAGS_OCF          = ['NetCashProvidedByUsedInOperatingActivities', 'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations']
TAGS_EQUITY_GAIN  = ['EquitySecuritiesFvNiGainLoss', 'MarketableSecuritiesGainLoss', 'UnrealizedGainLossOnInvestments']
TAGS_PURCHASE_OBL = ['LongTermPurchaseCommitmentAmount', 'UnconditionalPurchaseObligationBalanceSheetAmount', 'PurchaseObligation']
TAGS_LEASE_LIAB   = ['OperatingLeaseLiabilityNoncurrent']
TAGS_REVENUE      = ['RevenueFromContractWithCustomerExcludingAssessedTax', 'Revenues']
TAGS_GROSS_PROFIT = ['GrossProfit']
TAGS_INVENTORY    = ['InventoryNet']

print('Helpers ready.')


## Hyperscaler earnings-quality dashboard

For each hyperscaler: PP&E, implied useful-life trend (the depreciation test), capex run-rate, FCF proxy,
latest unrealized equity-securities gain/loss (the SpaceX-stake mechanism), and off-balance-sheet purchase
commitments. Everything trended so you can see direction, not just a snapshot.

In [ ]:
def hyperscaler_snapshot(ticker):
    cik = cik_for(ticker)
    if cik is None:
        return {'ticker': ticker, 'error': 'CIK not found'}

    out = {'ticker': ticker}

    # PP&E + depreciation -> implied useful life (years). Lengthening = the earnings-quality flag.
    # KNOWN QUIRK (confirmed live on GOOGL): companies sometimes stop tagging a concept with the same
    # XBRL element (e.g. switch to a disaggregated PP&E breakdown) — the tag goes quietly stale even
    # though the company keeps filing. Always check ppe_as_of vs dep_as_of; if they diverge by more
    # than ~120 days, the ratio below is comparing a STALE balance to a FRESH flow — flagged explicitly.
    _, ppe = get_concept_any(cik, TAGS_PPE)
    dep_tag, dep = get_concept_any(cik, TAGS_DEPRECIATION)
    if ppe is not None and dep is not None:
        dep_ann = latest_annualized(dep)
        ppe_latest = ppe.sort_values('end').iloc[-1]
        if dep_ann is not None and dep_ann[0]:
            out['ppe_net_latest_$B'] = round(ppe_latest['val'] / 1e9, 1)
            out['ppe_as_of'] = ppe_latest['end'].date().isoformat()
            out['dep_as_of'] = dep_ann[1].date().isoformat()
            staleness_days = abs((dep_ann[1] - ppe_latest['end']).days)
            useful_life = round(ppe_latest['val'] / dep_ann[0], 1)
            if staleness_days > 120:
                out['implied_useful_life_yrs'] = f"{useful_life} ⚠️ STALE PP&E TAG ({staleness_days}d gap vs dep — verify in the 10-Q text, don't trust this ratio as-is)"
            else:
                out['implied_useful_life_yrs'] = useful_life
        else:
            out['implied_useful_life_yrs'] = 'N/A (depreciation not annualizable)'
    else:
        out['implied_useful_life_yrs'] = f'N/A (dep tag: {dep_tag or "none found"})'

    # Capex run-rate (annualized)
    _, capex = get_concept_any(cik, TAGS_CAPEX)
    capex_ann = latest_annualized(capex) if capex is not None else None
    out['capex_annualized_$B'] = round(capex_ann[0] / 1e9, 1) if capex_ann else 'N/A'

    # FCF proxy = OCF - capex (both annualized off the same latest quarter where possible)
    _, ocf = get_concept_any(cik, TAGS_OCF)
    ocf_ann = latest_annualized(ocf) if ocf is not None else None
    if ocf_ann and capex_ann:
        out['fcf_proxy_annualized_$B'] = round((ocf_ann[0] - capex_ann[0]) / 1e9, 1)
    else:
        out['fcf_proxy_annualized_$B'] = 'N/A'

    # Unrealized equity-securities gain/loss — the SpaceX-stake catcher. Show the MOST RECENT single value, not annualized.
    eq_tag, eq = get_concept_any(cik, TAGS_EQUITY_GAIN)
    if eq is not None:
        row = eq.sort_values('end').iloc[-1]
        out['latest_equity_gain_$B'] = round(row['val'] / 1e9, 2)
        out['equity_gain_period_end'] = row['end'].date().isoformat()
        out['equity_gain_tag'] = eq_tag
    else:
        out['latest_equity_gain_$B'] = 'N/A (not tagged)'

    # Off-balance-sheet purchase commitments — the $811B/Beignet thread
    po_tag, po = get_concept_any(cik, TAGS_PURCHASE_OBL)
    if po is not None:
        row = po.sort_values('end').iloc[-1]
        out['purchase_commitments_$B'] = round(row['val'] / 1e9, 1)
        out['purchase_commitments_tag'] = po_tag
    else:
        out['purchase_commitments_$B'] = 'N/A (not tagged — check 10-K text/full-text search)'

    return out

print('Pulling hyperscaler snapshots (this hits several SEC endpoints per ticker, ~10-20s total)...\n')
hyperscaler_rows = []
for t in HYPERSCALERS:
    snap = hyperscaler_snapshot(t)
    hyperscaler_rows.append(snap)
    print(f"--- {t} ---")
    for k, v in snap.items():
        if k != 'ticker':
            print(f'  {k:28s}: {v}')
    print()

hyperscaler_df = pd.DataFrame(hyperscaler_rows)


## Secondaries fundamentals — revenue, margin, inventory (the memory/chip-cycle read)

In [ ]:
def secondary_snapshot(ticker):
    cik = cik_for(ticker)
    if cik is None:
        return {'ticker': ticker, 'error': 'CIK not found'}
    out = {'ticker': ticker}

    _, rev = get_concept_any(cik, TAGS_REVENUE)
    rev_ann = latest_annualized(rev) if rev is not None else None
    out['revenue_annualized_$B'] = round(rev_ann[0] / 1e9, 1) if rev_ann else 'N/A'

    _, gp = get_concept_any(cik, TAGS_GROSS_PROFIT)
    gp_ann = latest_annualized(gp) if gp is not None else None
    if gp_ann and rev_ann and rev_ann[0]:
        out['gross_margin_pct'] = round(100 * gp_ann[0] / rev_ann[0], 1)
    else:
        out['gross_margin_pct'] = 'N/A'

    _, inv = get_concept_any(cik, TAGS_INVENTORY)
    if inv is not None and len(inv) >= 2:
        inv_sorted = inv.sort_values('end')
        latest = inv_sorted.iloc[-1]
        prior_yr = inv_sorted[inv_sorted['end'] <= latest['end'] - pd.Timedelta(days=330)]
        out['inventory_latest_$B'] = round(latest['val'] / 1e9, 2)
        if not prior_yr.empty:
            yoy = 100 * (latest['val'] / prior_yr.iloc[-1]['val'] - 1)
            out['inventory_yoy_pct'] = round(yoy, 1)
        else:
            out['inventory_yoy_pct'] = 'N/A (no year-ago comp)'
    else:
        out['inventory_latest_$B'] = 'N/A'

    return out

print('Pulling secondaries snapshots...\n')
secondary_rows = []
for t in SECONDARIES + PERIPHERY:
    snap = secondary_snapshot(t)
    secondary_rows.append(snap)
    print(f"--- {t} ---")
    for k, v in snap.items():
        if k != 'ticker':
            print(f'  {k:22s}: {v}')
    print()

secondary_df = pd.DataFrame(secondary_rows)


## Ad-hoc full-text search — the "go find the receipt" tool

Search any keyword across recent SEC filings (10-K/10-Q/8-K, all companies or restricted to the watchlist).
This is the Schedule-D move: instead of waiting for a reporter to surface a footnote, search for it — a
counterparty name (`"CoreWeave"`, `"OpenAI"`, `"SpaceX"`), a structure (`"special purpose entity"`,
`"take-or-pay"`, `"guarantee"`), or anything else you're trying to verify.

In [ ]:
def search_edgar(keyword, forms='10-K,10-Q,8-K', start='2025-01-01', end=None, restrict_to_watchlist=True, max_results=15):
    """Full-text search across SEC EDGAR filings. Returns a DataFrame of hits."""
    if end is None:
        end = datetime.today().strftime('%Y-%m-%d')
    params = {
        'q': f'"{keyword}"',
        'forms': forms,
        'dateRange': 'custom',
        'startdt': start,
        'enddt': end,
    }
    d = _get('https://efts.sec.gov/LATEST/search-index', params=params)
    if d is None:
        print('⚠️ Search failed — check connection and try again.')
        return pd.DataFrame()

    total = d.get('hits', {}).get('total', {}).get('value', 0)
    hits = d.get('hits', {}).get('hits', [])
    rows = []
    watch_ciks = {cik_for(t) for t in ALL_TICKERS if cik_for(t)}
    for h in hits:
        src = h.get('_source', {})
        cik_list = src.get('ciks', [])
        hit_cik = cik_list[0].zfill(10) if cik_list else None
        if restrict_to_watchlist and hit_cik not in watch_ciks:
            continue
        rows.append({
            'company': ', '.join(src.get('display_names', [])),
            'form': ', '.join(src.get('forms', [])) if isinstance(src.get('forms'), list) else src.get('forms'),
            'filed': src.get('file_date'),
            'accession': h.get('_id'),
        })
        if len(rows) >= max_results:
            break
    print(f'Total hits for "{keyword}": {total} (showing {"watchlist-only" if restrict_to_watchlist else "all companies"}, up to {max_results})')
    return pd.DataFrame(rows)

# Example — try your own keywords by editing this cell:
example_results = search_edgar('SpaceX', restrict_to_watchlist=True)
example_results


## One-look summary — everything in one table

Run this last. Combines both dashboards and flags the loudest signals: a lengthening implied useful life
(the depreciation-schedule concern), a large fresh unrealized equity gain (the SpaceX-stake mechanism), and
large purchase commitments relative to the balance sheet.

In [ ]:
print('='*70)
print('HYPERSCALERS — earnings-quality dashboard')
print('='*70)
display_cols = ['ticker', 'implied_useful_life_yrs', 'capex_annualized_$B', 'fcf_proxy_annualized_$B',
                 'latest_equity_gain_$B', 'purchase_commitments_$B']
print(hyperscaler_df[[c for c in display_cols if c in hyperscaler_df.columns]].to_string(index=False))

print()
print('='*70)
print('SECONDARIES + PERIPHERY — fundamentals dashboard')
print('='*70)
display_cols2 = ['ticker', 'revenue_annualized_$B', 'gross_margin_pct', 'inventory_latest_$B', 'inventory_yoy_pct']
print(secondary_df[[c for c in display_cols2 if c in secondary_df.columns]].to_string(index=False))

print()
print('='*70)
print('Reminders:')
print('- "N/A (not tagged)" is common and expected — not every company tags every concept in XBRL.')
print('  Use the full-text search cell above to dig into the actual filing text instead.')
print('- Re-run the search cell with different keywords for ad-hoc digging (counterparty names,')
print('  "special purpose entity", "take-or-pay", "first-loss", etc).')
print('- This pulls what companies filed — it does not interpret it. Firewall discipline: treat')
print('  everything above as DATA for the vault, write the THESIS separately.')
